# NNCars BeginnerMix — 自動化實驗 (Colab)
Stage 1 presets → Stage 2 auto-tune → promote templates。
填入你自己的 repo URL，執行各 cell。

In [ ]:
REPO_URL = "https://github.com/sshuang610/NNCar_experiment.git"  # 你的新 repo
REPO_BRANCH = "main"
import os, subprocess, sys
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas"], check=True)

In [ ]:
import subprocess
out = subprocess.run([sys.executable, "-m", "pipeline.run_experiment",
                      "--config", "configs/presets/starter_presets.json"],
                     capture_output=True, text=True, check=True)
RUN_DIR = out.stdout.strip().splitlines()[-1]
print("run dir:", RUN_DIR)

In [ ]:
import pandas as pd
df = pd.read_csv(f"{RUN_DIR}/summary.csv")
df.sort_values(["validation_finish_count", "avg_finish_time", "avg_max_track_progress"],
               ascending=[False, True, False])

In [ ]:
import json
summary = json.load(open(f"{RUN_DIR}/summary.json"))
manifest = json.load(open(f"{RUN_DIR}/manifest.json"))
def rank(r):
    v = r["best_validation"]; ft = v["avg_finish_time"]
    return (v["finish_count"], -(ft if ft is not None else 1e9), v["avg_max_track_progress"])
winner = max([s for s in summary["strategies"]
              if s["strategy_name"] not in ("speed_only_baseline", "progress_only")], key=rank)
params = {s["name"]: s["params"] for s in manifest["strategies"]}[winner["strategy_name"]]
base_cfg = {**{k: manifest[k] for k in ("architecture","population_size","generations","mutation_rate",
              "train_seeds","validation_seeds","time_limit_seconds","fps","master_seed")},
            "run_name": "tune", "output_dir": "artifacts/runs", "max_seed_retries": 0,
            "parallel_workers": 6,
            "strategies": [{"name": "base", "strategy": "beginner_mix", "params": params}]}
os.makedirs("configs/tune", exist_ok=True)
json.dump(base_cfg, open("configs/tune/auto_base.json", "w"), indent=2)
# --out writes the FINAL tuned winner config (re-runnable) so the promote step below
# uses the auto-tuned recipe, not the pre-tune base.
subprocess.run([sys.executable, "-m", "pipeline.tune",
                "--base-config", "configs/tune/auto_base.json",
                "--out", "configs/tune/auto_winner.json",
                "--rounds", "2", "--step", "15"],
               check=True)

In [ ]:
# Re-run the FINAL tuned winner config to get a clean run dir, then promote it.
final = subprocess.run([sys.executable, "-m", "pipeline.run_experiment",
                        "--config", "configs/tune/auto_winner.json"],
                       capture_output=True, text=True, check=True)
FINAL_DIR = final.stdout.strip().splitlines()[-1]
from pipeline.export import promote_template
out_dir = promote_template(FINAL_DIR, "base", "winner_v1", group_id="0", username="ga_research")
print("template at:", out_dir)
print(open(f"{out_dir}/result.json").read())

In [ ]:
import shutil
shutil.make_archive("templates_export", "zip", "templates")
try:
    from google.colab import files; files.download("templates_export.zip")
except Exception:
    print("templates_export.zip ready in working dir")